# AI-OCR実験ノートブック

このノートブックでは、AI-OCRの機能を実験・検証できます。

## 主な機能
- Google Gemini / OpenAI を使用したOCR処理
- 画像・PDFファイルの構造化テキスト抽出
- 領収書・請求書の項目自動識別
- 信頼度スコアの評価

## 必要な環境変数
```bash
export GEMINI_API_KEY= # Google Gemini使用時
# または
export OPENAI_API_KEY=your_api_key_here  # OpenAI使用時
```


In [ ]:
# 必要なライブラリのインポート
import os
import base64
import logging
import time
from typing import Any, Dict, Optional, Type, Union, List, Literal
from pathlib import Path
from pydantic import BaseModel, Field
from enum import Enum
import json

# バックエンドの.envファイルからAPIキーのみを読み込み
try:
    from dotenv import load_dotenv
    # バックエンドの.envファイルを読み込み
    backend_env_path = Path("backend/.env")
    if backend_env_path.exists():
        load_dotenv(backend_env_path)
        print("✅ バックエンドの.envファイルを読み込みました")
    else:
        print("⚠️ backend/.envファイルが見つかりません")
        load_dotenv()  # 現在ディレクトリの.envを試行
except ImportError:
    print("⚠️ python-dotenvがインストールされていません: pip install python-dotenv")
    print("環境変数を直接設定してください")

# APIキーをハードコードで設定（一時的な解決策）
# 注意: 本番環境では環境変数を使用してください
GEMINI_API_KEY = ""  # ← ここに実際のAPIキーを入力してください
os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
print("🔑 APIキーをハードコードで設定しました")

# API接続の確認
print("\n🔧 環境設定チェック...")
print(f"GEMINI_API_KEY: {'✅ 設定済み' if os.getenv('GEMINI_API_KEY') else '❌ 未設定'}")
print(f"OPENAI_API_KEY: {'✅ 設定済み' if os.getenv('OPENAI_API_KEY') else '❌ 未設定'}")

# ログ設定
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


✅ バックエンドの.envファイルを読み込みました
🔑 APIキーをハードコードで設定しました

🔧 環境設定チェック...
GEMINI_API_KEY: ✅ 設定済み
OPENAI_API_KEY: ❌ 未設定


In [50]:
# 【シンプル2ステップOCR】スキーマ定義

# 【ステップ1】文字起こし結果
class TranscriptionResult(BaseModel):
    """ステップ1: 文字起こし結果"""
    full_text: str = Field(..., description="領収書全体の文字起こしテキスト")
    document_type: Optional[str] = Field(None, description="文書種別")

# 【ステップ2】領収書情報抽出結果  
class ReceiptInfo(BaseModel):
    """ステップ2: 領収書から抽出された情報"""
    # 基本情報
    store_name: Optional[str] = Field(None, description="店舗名・会社名")
    store_address: Optional[str] = Field(None, description="店舗住所") 
    store_phone: Optional[str] = Field(None, description="店舗電話番号")
    
    # 金額関連
    total_amount: Optional[str] = Field(None, description="合計金額")
    tax_rate: Optional[str] = Field(None, description="税率（%）")
    tax_amount: Optional[str] = Field(None, description="消費税額")
    subtotal_amount: Optional[str] = Field(None, description="税抜金額")
    
    # 日時情報
    issue_date: Optional[str] = Field(None, description="発行日（YYYY-MM-DD形式）")
    # 識別情報
    receipt_number: Optional[str] = Field(None, description="レシート番号・伝票番号")
    invoice_number: Optional[str] = Field(None, description="インボイス番号（T番号）")
    
    # 支払・商品情報
    payment_method: Optional[str] = Field(None, description="支払方法")
    items_summary: Optional[str] = Field(None, description="購入商品の概要")

print("✅ 2ステップOCRスキーマ定義完了")


✅ 2ステップOCRスキーマ定義完了


In [51]:
# 【共通設定】LLM設定
# 注：スキーマ定義はセル3で行われています

class LLMProvider(str, Enum):
    google = "google"
    openai = "openai"

class StructuredOutputConfig(BaseModel):
    provider: LLMProvider
    model: str
    max_output_tokens: Optional[int] = None
    temperature: Optional[float] = None
    retries: int = 3
    delay_sec: float = 2.0

print("✅ LLM設定完了")


✅ LLM設定完了


In [52]:
# LLMクライアントの初期化とヘルパー関数
# Optional import guards
try:
    from google import genai
    from google.genai import types as genai_types
    print("✅ Google Genai インポート成功")
except Exception as e:
    genai = None
    genai_types = None
    print(f"⚠️ Google Genai インポート失敗: {e}")

try:
    from openai import OpenAI
    print("✅ OpenAI インポート成功")
except Exception as e:
    OpenAI = None
    print(f"⚠️ OpenAI インポート失敗: {e}")

# クライアント管理
_GOOGLE_CLIENT = None
_OPENAI_CLIENT = None

def get_google_client():
    global _GOOGLE_CLIENT
    if _GOOGLE_CLIENT is None:
        if genai is None:
            raise RuntimeError("google-genai がインストールされていません。`pip install google-genai`")
        api_key = os.environ.get("GEMINI_API_KEY")
        if not api_key:
            raise RuntimeError("環境変数 GEMINI_API_KEY が設定されていません。")
        _GOOGLE_CLIENT = genai.Client(api_key=api_key)
    return _GOOGLE_CLIENT

def get_openai_client():
    global _OPENAI_CLIENT
    if _OPENAI_CLIENT is None:
        if OpenAI is None:
            raise RuntimeError("openai パッケージが見つかりません。`pip install openai`")
        if not os.environ.get("OPENAI_API_KEY"):
            raise RuntimeError("環境変数 OPENAI_API_KEY が設定されていません。")
        _OPENAI_CLIENT = OpenAI()
    return _OPENAI_CLIENT

print("✅ クライアント初期化関数準備完了")


✅ Google Genai インポート成功
✅ OpenAI インポート成功
✅ クライアント初期化関数準備完了


In [53]:
# コアOCR処理関数
def with_retries(func, max_retries: int, delay_sec: float, label: str) -> Any:
    """リトライ処理"""
    attempt = 0
    last_exc = None
    while attempt < max_retries:
        try:
            attempt += 1
            logger.info(f"{label} attempt {attempt}/{max_retries}")
            return func()
        except Exception as e:
            last_exc = e
            logger.warning(f"{label} failed attempt {attempt} with {e!r}")
            if attempt < max_retries:
                time.sleep(delay_sec)
    logger.error(f"{label} all {max_retries} attempts failed.")
    raise last_exc

def build_content_google(page_bytes: Optional[bytes], mime_type: Optional[str]):
    """Google用コンテンツ構築"""
    if page_bytes is None:
        return None
    if genai_types is None:
        raise RuntimeError("google-genai types の読み込みに失敗しました。")
    if not mime_type:
        mime_type = "application/octet-stream"
    return genai_types.Part.from_bytes(data=page_bytes, mime_type=mime_type)

def build_content_openai(page_bytes: Optional[bytes], file_name: Optional[str], mime_type: Optional[str]):
    """OpenAI用コンテンツ構築"""
    if page_bytes is None:
        return None
    mt = (mime_type or "application/octet-stream").lower()
    b64 = base64.b64encode(page_bytes).decode("utf-8")
    data_url = f"data:{mt};base64,{b64}"

    if mt == "application/pdf":
        if not file_name:
            file_name = "input.pdf"
        return {"type": "input_file", "filename": file_name, "file_data": data_url}
    elif mt.startswith("image/"):
        return {"type": "input_image", "image_url": data_url}
    else:
        return {"type": "input_file", "filename": file_name or "input.bin", "file_data": data_url}

print("✅ ヘルパー関数準備完了")


✅ ヘルパー関数準備完了


In [54]:
# メインOCR処理関数
def generate_structured_output(
    config: StructuredOutputConfig,
    prompt: str,
    response_schema: Type[BaseModel],
    *,
    content_bytes: Optional[bytes] = None,
    mime_type: Optional[str] = None,
    file_name: Optional[str] = None,
) -> BaseModel:
    """構造化出力生成のメイン関数"""
    provider = config.provider.value if isinstance(config.provider, Enum) else config.provider

    if provider == LLMProvider.google.value:
        client = get_google_client()
        content_part = build_content_google(content_bytes, mime_type)
        contents = []
        if content_part is not None:
            contents.append(content_part)
        contents.append(prompt)

        def _call():
            return client.models.generate_content(
                model=config.model,
                contents=contents,
                config={
                    "response_mime_type": "application/json",
                    "response_schema": response_schema,
                },
            )

        resp = with_retries(_call, config.retries, config.delay_sec, label="google-genai")
        parsed = getattr(resp, "parsed", None)
        if parsed is None:
            raise RuntimeError("google-genai: レスポンスの parsed を取得できませんでした。")
        return parsed

    elif provider == LLMProvider.openai.value:
        client = get_openai_client()
        user_inputs = []
        content_obj = build_content_openai(content_bytes, file_name, mime_type)
        if content_obj is not None:
            user_inputs.append(content_obj)
        user_inputs.append({"type": "input_text", "text": prompt})

        def _call():
            return client.responses.parse(
                model=config.model,
                max_output_tokens=config.max_output_tokens,
                temperature=config.temperature,
                input=[{"role": "user", "content": user_inputs}],
                text_format=response_schema,
            )

        resp = with_retries(_call, config.retries, config.delay_sec, label="openai")
        parsed = getattr(resp, "output_parsed", None)
        if parsed is None:
            raise RuntimeError("openai: output_parsed を取得できませんでした。")
        return parsed

    else:
        raise ValueError(f"Unsupported provider: {config.provider}")

print("✅ メインOCR処理関数準備完了")


✅ メインOCR処理関数準備完了


In [ ]:
# 【シンプル2ステップOCR】処理関数

# ステップ1: 文字起こし用プロンプト
TRANSCRIPTION_PROMPT = """あなたは領収書専用のOCRエンジンです。
入力された領収書画像から文字を読み取り、全体のテキストを正確に抽出してください。
また位置や構造も含めて抽出してください。

【重要】
- 文字が不鮮明でも可能な限り推定して抽出
- 全体のテキストを漏れなく抽出
- 日本語の文字認識に特に注意を払う"""

def step1_transcription(
    config: StructuredOutputConfig,
    content_bytes: bytes,
    mime_type: str,
    file_name: str = None,
) -> TranscriptionResult:
    """ステップ1: 文字起こし"""
    print("🔤 ステップ1: 文字起こし処理中...")
    
    result = generate_structured_output(
        config=config,
        prompt=TRANSCRIPTION_PROMPT,
        response_schema=TranscriptionResult,
        content_bytes=content_bytes,
        mime_type=mime_type,
        file_name=file_name,
    )
    
    print(f"✅ 文字起こし完了 ({len(result.full_text)}文字)")
    return result

# ステップ2: 情報抽出用プロンプト  
EXTRACTION_PROMPT = """以下の領収書テキストから必要な情報を抽出してください。

【抽出項目】
- 店舗名、住所、電話番号
- 合計金額、税率、消費税額、税抜金額  
- 発行日（YYYY-MM-DD形式）、発行時刻（HH:MM形式）
- レシート番号、インボイス番号
- 支払方法、商品概要

【抽出指針】
- 見つからない項目は null に設定
- 金額は数値のみ（¥マークや,は除去）
- 日付は統一形式で

【領収書テキスト】
{text}

上記から情報を抽出してください。"""

def step2_extraction(
    config: StructuredOutputConfig,
    transcription: TranscriptionResult,
) -> ReceiptInfo:
    """ステップ2: 情報抽出"""
    print("🎯 ステップ2: 情報抽出処理中...")
    
    formatted_prompt = EXTRACTION_PROMPT.format(text=transcription.full_text)
    
    result = generate_structured_output(
        config=config,
        prompt=formatted_prompt,
        response_schema=ReceiptInfo,
        content_bytes=None,
        mime_type=None,
        file_name=None,
    )
    
    print("✅ 情報抽出完了")
    return result

print("✅ シンプル2ステップOCR関数準備完了")


✅ シンプル2ステップOCR関数準備完了


In [56]:
# 【シンプル結果表示】関数

def print_results(transcription: TranscriptionResult, receipt_info: ReceiptInfo):
    """結果を見やすく表示"""
    print("\n" + "="*50)
    print("🔍 シンプル2ステップOCR結果")
    print("="*50)
    
    # ステップ1結果
    print(f"\n📝 【ステップ1】文字起こし結果:")
    print(f"   文書種別: {transcription.document_type or '不明'}")
    print(f"   文字数: {len(transcription.full_text)}")
    print(f"\n📄 全体テキスト:")
    print("-" * 30)
    print(transcription.full_text)
    print("-" * 30)
    
    # ステップ2結果
    print(f"\n🎯 【ステップ2】抽出情報:")
    
    # 基本情報
    if receipt_info.store_name or receipt_info.store_address or receipt_info.store_phone:
        print("   【店舗情報】")
        if receipt_info.store_name:
            print(f"     店舗名: {receipt_info.store_name}")
        if receipt_info.store_address:
            print(f"     住所: {receipt_info.store_address}")
        if receipt_info.store_phone:
            print(f"     電話: {receipt_info.store_phone}")
    
    # 金額情報
    if receipt_info.total_amount or receipt_info.tax_rate or receipt_info.tax_amount or receipt_info.subtotal_amount:
        print("   【金額情報】")
        if receipt_info.total_amount:
            print(f"     合計金額: {receipt_info.total_amount}")
        if receipt_info.tax_rate:
            print(f"     税率: {receipt_info.tax_rate}")
        if receipt_info.tax_amount:
            print(f"     消費税: {receipt_info.tax_amount}")
        if receipt_info.subtotal_amount:
            print(f"     税抜金額: {receipt_info.subtotal_amount}")
    
    # 日時情報
    if receipt_info.issue_date or receipt_info.issue_time:
        print("   【日時情報】")
        if receipt_info.issue_date:
            print(f"     発行日: {receipt_info.issue_date}")
    
    # その他情報
    other_info = []
    if receipt_info.receipt_number:
        other_info.append(f"レシート番号: {receipt_info.receipt_number}")
    if receipt_info.invoice_number:
        other_info.append(f"インボイス番号: {receipt_info.invoice_number}")
    if receipt_info.payment_method:
        other_info.append(f"支払方法: {receipt_info.payment_method}")
    if receipt_info.items_summary:
        other_info.append(f"商品: {receipt_info.items_summary}")
    
    if other_info:
        print("   【その他】")
        for info in other_info:
            print(f"     {info}")
    
    print("\n" + "="*50)

print("✅ シンプル表示関数準備完了")


✅ シンプル表示関数準備完了


In [57]:
# 設定とユーティリティ関数
def get_ocr_config(provider: str = "google", model: str = None) -> StructuredOutputConfig:
    """OCR用のLLM設定を取得"""
    if provider == "google":
        default_model = "gemini-2.5-flash"
        return StructuredOutputConfig(
            provider=LLMProvider.google,
            model=model or default_model,
            retries=3,
            delay_sec=2.0
        )
    elif provider == "openai":
        default_model = "gpt-4o-2024-08-06"
        return StructuredOutputConfig(
            provider=LLMProvider.openai,
            model=model or default_model,
            max_output_tokens=4000,
            temperature=0.1,
            retries=3,
            delay_sec=2.0
        )
    else:
        raise ValueError(f"Unsupported provider: {provider}")

def detect_mime_type(file_path: str) -> str:
    """ファイル拡張子からMIMEタイプを推定"""
    ext = file_path.lower().split('.')[-1]
    mime_types = {
        'jpg': 'image/jpeg',
        'jpeg': 'image/jpeg',
        'png': 'image/png',
        'gif': 'image/gif',
        'pdf': 'application/pdf',
        'bmp': 'image/bmp',
        'webp': 'image/webp'
    }
    return mime_types.get(ext, 'application/octet-stream')

def check_file_exists(file_path: str):
    """ファイルの存在確認"""
    if Path(file_path).exists():
        file_size = Path(file_path).stat().st_size
        print(f"📁 ファイル確認: {Path(file_path).name} ({file_size:,} bytes)")
        return True
    else:
        print(f"❌ ファイルが見つかりません: {file_path}")
        return False

print("✅ ユーティリティ関数準備完了")


✅ ユーティリティ関数準備完了


In [ ]:
# 【ステップ3】金額計算・補完関数

def calculate_missing_amounts(receipt_info: ReceiptInfo) -> ReceiptInfo:
    """
    抽出結果の金額項目をチェックし、不足している消費税・税抜金額を計算
    
    Args:
        receipt_info: ステップ2で抽出された領収書情報
        
    Returns:
        計算結果で補完された領収書情報
    """
    print("🧮 ステップ3: 金額計算・補完処理中...")
    
    # 金額関連項目の状態をチェック
    has_total = receipt_info.total_amount is not None
    has_tax_rate = receipt_info.tax_rate is not None  
    has_tax_amount = receipt_info.tax_amount is not None
    has_subtotal = receipt_info.subtotal_amount is not None
    
    print(f"   📊 現在の状態:")
    print(f"     合計金額: {'✅' if has_total else '❌'} {receipt_info.total_amount or 'null'}")
    print(f"     税率: {'✅' if has_tax_rate else '❌'} {receipt_info.tax_rate or 'null'}")
    print(f"     消費税: {'✅' if has_tax_amount else '❌'} {receipt_info.tax_amount or 'null'}")
    print(f"     税抜金額: {'✅' if has_subtotal else '❌'} {receipt_info.subtotal_amount or 'null'}")
    
    # 全項目が揃っている場合はそのまま返す
    if has_total and has_tax_rate and has_tax_amount and has_subtotal:
        print("   ✅ 全項目が揃っています。計算をスキップします。")
        return receipt_info
    
    # 計算に必要な項目（合計金額・税率）がない場合は計算不可
    if not has_total or not has_tax_rate:
        print("   ⚠️ 合計金額または税率が不足しているため、計算できません。")
        return receipt_info
    
    try:
        # 文字列から数値に変換
        total_str = receipt_info.total_amount.replace(',', '').replace('¥', '').replace('円', '').strip()
        tax_rate_str = receipt_info.tax_rate.replace('%', '').strip()
        
        total_amount = float(total_str)
        tax_rate = float(tax_rate_str) / 100  # パーセントを小数に変換
        
        print(f"   🔢 計算用数値:")
        print(f"     合計金額: {total_amount:,.0f}")
        print(f"     税率: {tax_rate:.1%}")
        
        # 消費税が不足している場合は計算
        if not has_tax_amount:
            # 消費税 = 合計金額 × 税率 ÷ (1 + 税率)
            calculated_tax = total_amount * tax_rate / (1 + tax_rate)
            receipt_info.tax_amount = str(int(round(calculated_tax)))
            print(f"   ➕ 消費税を計算: {calculated_tax:,.0f} → {receipt_info.tax_amount}")
        
        # 税抜金額が不足している場合は計算
        if not has_subtotal:
            # 税抜金額 = 合計金額 - 消費税
            tax_amount_num = float(receipt_info.tax_amount.replace(',', '').replace('¥', '').replace('円', '').strip())
            calculated_subtotal = total_amount - tax_amount_num
            receipt_info.subtotal_amount = str(int(round(calculated_subtotal)))
            print(f"   ➖ 税抜金額を計算: {calculated_subtotal:,.0f} → {receipt_info.subtotal_amount}")
        
        print("   ✅ 金額計算・補完完了")
        
    except (ValueError, AttributeError) as e:
        print(f"   ❌ 計算エラー: {e}")
        print("   数値形式が正しくないため、計算をスキップします。")
    
    return receipt_info

def print_calculation_summary(original_info: ReceiptInfo, calculated_info: ReceiptInfo):
    """計算前後の比較を表示"""
    print("\n" + "="*50)
    print("🧮 金額計算・補完結果")
    print("="*50)
    
    fields = [
        ("合計金額", original_info.total_amount, calculated_info.total_amount),
        ("税率", original_info.tax_rate, calculated_info.tax_rate),
        ("消費税", original_info.tax_amount, calculated_info.tax_amount),
        ("税抜金額", original_info.subtotal_amount, calculated_info.subtotal_amount),
    ]
    
    for field_name, original, calculated in fields:
        if original != calculated:
            print(f"   {field_name}: {original or 'null'} → {calculated or 'null'} ✨")
        else:
            print(f"   {field_name}: {calculated or 'null'}")
    
    print("="*50)

print("✅ 金額計算・補完関数準備完了")


✅ 金額計算・補完関数準備完了


## 🧪 シンプル2ステップOCR実験

以下のセルで実際にOCR処理を実行できます。

### 実験手順
1. 領収書画像ファイルのパスを指定
2. OCR処理を実行（2ステップで処理）
3. 結果を確認

### サンプルファイルの準備
領収書や請求書の画像ファイルを用意してください：
- 対応形式: JPG, PNG, PDF, GIF, BMP, WebP
- 推奨解像度: 300dpi以上
- ファイルサイズ: 10MB以下


In [63]:
# 【金額計算機能付き】シンプル3ステップOCR実験

# 設定
FILE_PATH = "C:\\Users\\zente\\Downloads\\1000001510.jpg" 
PROVIDER = "google"  
MODEL = "gemini-2.5-flash"  

print("🚀 シンプル3ステップOCR実験（金額計算機能付き）")
print(f"📁 ファイル: {FILE_PATH}")
print(f"🤖 {PROVIDER} / {MODEL}")

try:
    # ファイル確認・読み込み
    if not check_file_exists(FILE_PATH):
        print("❌ ファイルが見つかりません。FILE_PATHを確認してください。")
    else:
        with open(FILE_PATH, 'rb') as f:
            content_bytes = f.read()
        
        mime_type = detect_mime_type(FILE_PATH)
        file_name = Path(FILE_PATH).name
        config = get_ocr_config(PROVIDER, MODEL)
        
        print(f"📊 ファイルサイズ: {len(content_bytes):,} bytes")
        print()
        
        # ステップ1: 文字起こし
        transcription = step1_transcription(
            config=config,
            content_bytes=content_bytes,
            mime_type=mime_type,
            file_name=file_name
        )
        
        # ステップ2: 情報抽出
        receipt_info = step2_extraction(
            config=config,
            transcription=transcription
        )
        
        # ステップ3: 金額計算・補完（元の情報をコピーして保持）
        import copy
        original_receipt_info = copy.deepcopy(receipt_info)
        calculated_receipt_info = calculate_missing_amounts(receipt_info)
        
        # 結果表示（計算後の結果）
        print_results(transcription, calculated_receipt_info)
        
        # 計算結果の比較表示
        print_calculation_summary(original_receipt_info, calculated_receipt_info)
        
        print("\n" + "="*60)
        print("📋 最終結果確認（計算後）")
        print("="*60)
        
        # 最終結果の詳細表示
        print("\n🎯 【最終結果】抽出・計算後の情報:")
        
        # 全フィールドを確認（計算後）
        fields = [
            ("店舗名", calculated_receipt_info.store_name),
            ("住所", calculated_receipt_info.store_address),
            ("電話番号", calculated_receipt_info.store_phone),
            ("合計金額", calculated_receipt_info.total_amount),
            ("税率", calculated_receipt_info.tax_rate),
            ("消費税", calculated_receipt_info.tax_amount),
            ("税抜金額", calculated_receipt_info.subtotal_amount),
            ("発行日", calculated_receipt_info.issue_date),
            ("発行時刻", calculated_receipt_info.issue_time),
            ("レシート番号", calculated_receipt_info.receipt_number),
            ("インボイス番号", calculated_receipt_info.invoice_number),
            ("支払方法", calculated_receipt_info.payment_method),
            ("商品概要", calculated_receipt_info.items_summary),
        ]
        
        detected = 0
        for field_name, value in fields:
            if value:
                print(f"   ✅ {field_name}: {value}")
                detected += 1
            else:
                print(f"   ❌ {field_name}: null")
        
        print(f"\n   📊 最終抽出成功率: {detected}/{len(fields)} ({detected/len(fields)*100:.1f}%)")
        
        print("✅ 3ステップ処理完了！")
        
except Exception as e:
    print(f"❌ エラー: {e}")
    import traceback
    traceback.print_exc()


INFO:__main__:google-genai attempt 1/3
INFO:google_genai.models:AFC is enabled with max remote calls: 10.


🚀 シンプル3ステップOCR実験（金額計算機能付き）
📁 ファイル: C:\Users\zente\Downloads\1000001510.jpg
🤖 google / gemini-2.5-flash
📁 ファイル確認: 1000001510.jpg (744,744 bytes)
📊 ファイルサイズ: 744,744 bytes

🔤 ステップ1: 文字起こし処理中...


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC remote call 1 is done.
INFO:__main__:google-genai attempt 1/3
INFO:google_genai.models:AFC is enabled with max remote calls: 10.


✅ 文字起こし完了 (195文字)
🎯 ステップ2: 情報抽出処理中...


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC remote call 1 is done.


✅ 情報抽出完了
🧮 ステップ3: 金額計算・補完処理中...
   📊 現在の状態:
     合計金額: ✅ 9360
     税率: ✅ 10
     消費税: ❌ null
     税抜金額: ❌ null
   🔢 計算用数値:
     合計金額: 9,360
     税率: 10.0%
   ➕ 消費税を計算:calculated_tax = total_amount * tax_rate / (1 + tax_rate) 
   ➖ 税抜金額を計算: 8,509 → 8509
   ✅ 金額計算・補完完了

🔍 シンプル2ステップOCR結果

📝 【ステップ1】文字起こし結果:
   文書種別: 領収証
   文字数: 195

📄 全体テキスト:
------------------------------
領収証 様 No. ¥9,360- 但 2025年 9月19日 上記正に領収いたしました 登録番号 T6010801022509 内訳 税率 金額(税抜・税込) 10 % 消費税額等 税率 金額(税抜・税込) % 消費税額等 収入印紙 コクヨ ウケ-1097 おみっちゃん 恵比寿店 〒150-0022 東京都渋谷区恵比寿南1-3-9 新井ビル1F TEL・FAX 03-3713-2838
------------------------------

🎯 【ステップ2】抽出情報:
   【店舗情報】
     店舗名: おみっちゃん 恵比寿店
     住所: 〒150-0022 東京都渋谷区恵比寿南1-3-9 新井ビル1F
     電話: 03-3713-2838
   【金額情報】
     合計金額: 9360
     税率: 10
     消費税: 851
     税抜金額: 8509
   【日時情報】
     発行日: 2025-09-19
   【その他】
     インボイス番号: T6010801022509


🧮 金額計算・補完結果
   合計金額: 9360
   税率: 10
   消費税: null → 851 ✨
   税抜金額: null → 8509 ✨

📋 最終結果確認（計算後）

🎯 【最終結果】抽出・計算後の情報:
❌ エラー: 'ReceiptInfo' object has no 

Traceback (most recent call last):
  File "C:\Users\zente\AppData\Local\Temp\ipykernel_23712\1951901843.py", line 69, in <module>
    ("発行時刻", calculated_receipt_info.issue_time),
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\zente\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\main.py", line 828, in __getattr__
    raise AttributeError(f'{type(self).__name__!r} object has no attribute {item!r}')
AttributeError: 'ReceiptInfo' object has no attribute 'issue_time'


In [61]:
# 設定
FILE_PATH = "C:\\Users\\zente\\Downloads\\1000001510.jpg" 
PROVIDER = "google"  
MODEL = "gemini-2.5-flash"  

print("🚀 シンプル2ステップOCR実験")
print(f"📁 ファイル: {FILE_PATH}")
print(f"🤖 {PROVIDER} / {MODEL}")

try:
    # ファイル確認・読み込み
    if not check_file_exists(FILE_PATH):
        print("❌ ファイルが見つかりません。FILE_PATHを確認してください。")
    else:
        with open(FILE_PATH, 'rb') as f:
            content_bytes = f.read()
        
        mime_type = detect_mime_type(FILE_PATH)
        file_name = Path(FILE_PATH).name
        config = get_ocr_config(PROVIDER, MODEL)
        
        print(f"📊 ファイルサイズ: {len(content_bytes):,} bytes")
        print()
        
        # ステップ1: 文字起こし
        transcription = step1_transcription(
            config=config,
            content_bytes=content_bytes,
            mime_type=mime_type,
            file_name=file_name
        )
        
        # ステップ2: 情報抽出
        receipt_info = step2_extraction(
            config=config,
            transcription=transcription
        )
        
        # 結果表示（従来の表示）
        print_results(transcription, receipt_info)
        
        print("\n" + "="*60)
        print("📋 詳細結果確認（null項目も含む）")
        print("="*60)
        
        # ステップ1結果の詳細表示
        print("\n🔤 【ステップ1】文字起こし結果:")
        print(f"   文書種別: {transcription.document_type or 'null'}")
        print(f"   全体文字数: {len(transcription.full_text)}")
        print("\n   📄 抽出されたテキスト:")
        print("   " + "-" * 40)
        # テキストを行ごとに表示（見やすくするため）
        for line in transcription.full_text.split('\n'):
            if line.strip():  # 空行でない場合のみ表示
                print(f"   {line}")
        print("   " + "-" * 40)
        
        # ステップ2結果の詳細表示（null項目も含む）
        print("\n🎯 【ステップ2】抽出情報の詳細（null項目も表示）:")
        
        # 全フィールドを確認
        fields = [
            ("店舗名", receipt_info.store_name),
            ("住所", receipt_info.store_address),
            ("電話番号", receipt_info.store_phone),
            ("合計金額", receipt_info.total_amount),
            ("税率", receipt_info.tax_rate),
            ("消費税", receipt_info.tax_amount),
            ("税抜金額", receipt_info.subtotal_amount),
            ("発行日", receipt_info.issue_date),
            ("レシート番号", receipt_info.receipt_number),
            ("インボイス番号", receipt_info.invoice_number),
            ("支払方法", receipt_info.payment_method),
            ("商品概要", receipt_info.items_summary),
        ]
        
        detected = 0
        for field_name, value in fields:
            if value:
                print(f"   ✅ {field_name}: {value}")
                detected += 1
            else:
                print(f"   ❌ {field_name}: null")
        
        print(f"\n   📊 抽出成功率: {detected}/{len(fields)} ({detected/len(fields)*100:.1f}%)")
        
        print("✅ 処理完了！")
        
except Exception as e:
    print(f"❌ エラー: {e}")
    import traceback
    traceback.print_exc()


INFO:__main__:google-genai attempt 1/3
INFO:google_genai.models:AFC is enabled with max remote calls: 10.


🚀 シンプル2ステップOCR実験
📁 ファイル: C:\Users\zente\Downloads\1000001510.jpg
🤖 google / gemini-2.5-flash
📁 ファイル確認: 1000001510.jpg (744,744 bytes)
📊 ファイルサイズ: 744,744 bytes

🔤 ステップ1: 文字起こし処理中...


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC remote call 1 is done.
INFO:__main__:google-genai attempt 1/3
INFO:google_genai.models:AFC is enabled with max remote calls: 10.


✅ 文字起こし完了 (198文字)
🎯 ステップ2: 情報抽出処理中...


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC remote call 1 is done.


✅ 情報抽出完了

🔍 シンプル2ステップOCR結果

📝 【ステップ1】文字起こし結果:
   文書種別: 領収証
   文字数: 198

📄 全体テキスト:
------------------------------
領収証 様 No. ¥9.360- 但 2025 年 9 月 19 日 上記正に領収いたしました 登録番号 T6010801022509 内訳 税率 金額(税抜・税込) 10 % 消費税額等 税率 金額(税抜・税込) % 消費税額等 収入印紙 コクヨ ウケ-1097 おみっちゃ 恵比寿店 〒150-0022 東京都渋谷区恵比寿南1-3-9 新井ビル1F TEL・FAX 03-3713-2838
------------------------------

🎯 【ステップ2】抽出情報:
   【店舗情報】
     店舗名: おみっちゃ 恵比寿店
     住所: 〒150-0022 東京都渋谷区恵比寿南1-3-9 新井ビル1F
     電話: 03-3713-2838
   【金額情報】
     合計金額: 9360
     税率: 10
   【日時情報】
     発行日: 2025-09-19
   【その他】
     インボイス番号: T6010801022509


📋 詳細結果確認（null項目も含む）

🔤 【ステップ1】文字起こし結果:
   文書種別: 領収証
   全体文字数: 198

   📄 抽出されたテキスト:
   ----------------------------------------
   領収証 様 No. ¥9.360- 但 2025 年 9 月 19 日 上記正に領収いたしました 登録番号 T6010801022509 内訳 税率 金額(税抜・税込) 10 % 消費税額等 税率 金額(税抜・税込) % 消費税額等 収入印紙 コクヨ ウケ-1097 おみっちゃ 恵比寿店 〒150-0022 東京都渋谷区恵比寿南1-3-9 新井ビル1F TEL・FAX 03-3713-2838
   ----------------------------------------

🎯 【ステップ2】抽出情報の詳細（null項目も表示）:
   ✅ 店舗名: おみっちゃ 恵比寿店
   ✅ 住所: 〒150